In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import pickle as pkl
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
#os.environ['TOKENIZERS_PARALLELISM'] = 'false' # there might be interferences with the parallelism of the Hugging Face Trainer
#os.environ['WANDB_DISABLED'] = "true"

#print(f"Using device: {device}"#)
#if torch.device.type == 'cuda':
    #print(torch.cuda.get_device_name(0))


Using device: cuda:0


In [2]:
#k_fold_test_file_name='/content/drive/MyDrive/data/independent_file.xlsx'
k_fold_test_file_name='data/independent_file.xlsx'
#k_fold_test_file_name='data/k_fold_data.xlsx'

In [3]:
kf_df = pd.read_excel(k_fold_test_file_name, index_col=0)
kf_df.head()
kf_df.shape

(1555, 9)

In [4]:
print("Unique labels/partys in dataset:{}".format(kf_df['party'].unique()))

Unique labels/partys in dataset:[9]


In [5]:
speech_party_map= dict(zip(kf_df['speechnumber'],kf_df['party']))

In [6]:
"""takes a text and returns a list of texts in given length"""
def chunksspeech(text,term, length):
    return [' '.join(chunk) for chunk in list((text[0+i:length+i] for i in range(0, int(term), length)))]

In [7]:
def chunk_data(texts,labels,speech_ids,length):
    chunked_speeches=[chunksspeech(text.split(" "),len(text.split(" ")),length) for text in texts]
    speech_chunks = [chunk for speech in chunked_speeches for chunk in speech]
    chunk_labels = [label for label,speech in zip(labels,chunked_speeches) for _ in speech]
    chunk_to_speech_mapping = [speech_id for speech_id,speech in zip(speech_ids,chunked_speeches) for _ in speech]
    return speech_chunks,chunk_labels,chunk_to_speech_mapping

In [8]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, ids, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.ids = ids
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        speech_id = self.ids[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label,dtype=torch.long), 'speech_id': speech_id}

In [9]:
'''
BERT Classifier with a BERT layer, Dropout layer and a linear layer
'''
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits

In [10]:
def evaluate(model, data_loader, device, ids=None):
    model.eval()
    predictions = []
    actual_labels = []
    all_probs = []
    all_ids = []  # collect speech Id to aggregate later
    with torch.no_grad():
        for batch in tqdm(data_loader,desc="Evaluation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            probs = nn.functional.softmax(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
            all_ids.extend(batch['speech_id'])  # edit to process chunks#'speech_id'
    return accuracy_score(actual_labels, predictions),classification_report(actual_labels, predictions), all_probs, all_ids #classification_report(actual_labels, predictions)

In [11]:
def predict_party(text, model, tokenizer, device, max_length=128):
    model.eval()
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)

    return preds.item()

In [12]:
"""Creates a dataframe with speech_id and predicitions on Chunk-level and aggregates to speech-level """

def speech_chunk_dataframe(prob, speech_ids):
    prob_df = pd.DataFrame(prob)
    speech_ids = [x.item() for x in speech_ids]
    prob_df['speechnumber'] = speech_ids
    return prob_df

"Group chunk-level probabilites by speech_id and compute weighted mean if provided"

def group_probablities_by_speech(probabilities_df, weights=None):
  grouped_probabilities = probabilities_df.groupby('speechnumber').mean()
  return grouped_probabilities

'''Evaluate on speech level'''

def evaluate_speeches(probabilities_df, speech_id_to_party_map, weights=None):
    grouped_probabilities = group_probablities_by_speech(probabilities_df, weights)
    # get highest average probability per speech
    predicted_speech_labels = grouped_probabilities.idxmax(axis=1).tolist()
    # get true labels
    grouped_speech_ids = grouped_probabilities.index.tolist()
    true_labels= [speech_id_to_party_map[k] for k in grouped_speech_ids]
    # get accuracy and report
    accuracy = accuracy_score(true_labels, predicted_speech_labels)
    report = classification_report(true_labels, predicted_speech_labels)

    return  accuracy, report,true_labels, predicted_speech_labels,grouped_probabilities

In [13]:
# Set up parameters
bert_model_name= 'bert-base-german-cased'
num_classes = 9
max_length = 256
batch_size = 20
num_epochs = 4
learning_rate = 2e-5
output_dir ='data/run'+ str(1)

In [16]:
#PATH = "/content/drive/MyDrive/data/bert_classifier_k_fold_5.pth"
PATH ="data/bert_classifier_k_fold_5.pth"
model = BERTClassifier(bert_model_name, num_classes)#.to(device)
state_dict =torch.load(PATH, weights_only=False)
model.load_state_dict(state_dict, strict=False)

Some weights of the model checkpoint at bert-base-german-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
# Set up parameters
bert_model_name= 'bert-base-german-cased'
num_classes = 9
max_length = 256
batch_size = 20
num_epochs = 2
learning_rate = 2e-5
tokenizer = BertTokenizer.from_pretrained(bert_model_name)

In [ ]:
#df_val = kf_df[kf_df['k_fold']==1]
#print(df_val.shape)
df_val = kf_df
speech_party_map= dict(zip(df_val['speechnumber'],df_val['party']))

v_texts=list(df_val['text'])
v_labels=list(df_val['party'])
v_mps=list(df_val['speaker'])
v_speech_ids=list(df_val['speechnumber'])

val_texts, val_labels, val_ids = chunk_data(v_texts,v_labels,v_speech_ids,max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, val_ids, tokenizer, max_length)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

accuracy, report, probabilities, speech_ids = evaluate(model, val_dataloader, device)

Evaluation:   0%|          | 0/194 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
pkl.dump(accuracy, open('/content/drive/MyDrive/data/accuracy_10kf_1.pkl', 'wb'))
pkl.dump(report, open('/content/drive/MyDrive/data/report_10kf_1.pkl', 'wb'))
pkl.dump(probabilities, open('/content/drive/MyDrive/data/probabilities_10kf_1.pkl', 'wb'))
pkl.dump(speech_ids, open('/content/drive/MyDrive/data/speech_ids_10kf_1.pkl', 'wb'))



In [ ]:
pkl.dump(accuracy, open('/content/drive/MyDrive/data/accuracy_ind_2.pkl', 'wb'))
pkl.dump(report, open('/content/drive/MyDrive/data/report_ind_2.pkl', 'wb'))
pkl.dump(probabilities, open('/content/drive/MyDrive/data/probabilities_ind_2".pkl', 'wb'))
pkl.dump(speech_ids, open('/content/drive/MyDrive/data/speech_ids_ind_2".pkl', 'wb'))

In [ ]:
# Aggregate on speech-level
speech_level_df = speech_chunk_dataframe(probabilities, speech_ids)
speech_accuracy, speech_report, true_speech_labels, predicted_speech_labels, speech_probabilities = evaluate_speeches(speech_level_df, speech_party_map, weights=None)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
full_eval= pd.merge(df_val, speech_probabilities, on= "speechnumber", how="outer")
full_eval['predicted_party']= predicted_speech_labels
full_eval.to_excel("/content/drive/MyDrive/data/df_val_independent_5.xlsx")

In [ ]:
print(report)

              precision    recall  f1-score   support

           0       0.48      0.61      0.54      5872
           1       0.71      0.62      0.66      7124
           2       0.46      0.57      0.51      4801
           3       0.70      0.49      0.58      6182
           4       0.28      0.26      0.27       342
           5       0.25      0.30      0.27      1085
           6       0.35      0.32      0.34       622
           7       0.07      0.06      0.07       235
           8       0.00      0.00      0.00       116

    accuracy                           0.55     26379
   macro avg       0.37      0.36      0.36     26379
weighted avg       0.57      0.55      0.55     26379

